In [ ]:

from matplotlib import pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs
from matplotlib.colors import ListedColormap


In [ ]:

# Generate some toy data
X, y = make_blobs(n_samples=[10, 15, 25], n_features=2, centers=[[-1, -1], [0, 1], [1, 0]])
X, y


In [ ]:

# Plot initial data
rgb_map = ListedColormap(('r', 'g', 'b'))
plt.grid(True); plt.axhline(0.0, c='k'); plt.axvline(0.0, c='k')
plt.legend(*plt.scatter(X[:, 0], X[:, 1], c=y, cmap=rgb_map, edgecolors='w').legend_elements(), title='Classes:')
plt.title('$y(x)$')
plt.axis('square'); plt.axis((-4.0, 4.0, -4.0, 4.0))
plt.show()


In [ ]:

# SMOTE Implementation
def smote(X, y, k_neighbors=5):
    classes, counts = np.unique(y, return_counts=True)
    n_per_class = counts.max()
    X_aug, y_aug = [], []
    for c, count in zip(classes, counts):
        instances = np.flatnonzero(y == c)
        for _ in range(n_per_class - count):
            instance = np.random.choice(instances)
            distances = np.sum((X[instances, :] - X[instance, :]) ** 2, axis=1)
            neighbors = instances[np.argpartition(distances, k_neighbors+1)[1: k_neighbors+1]]
            neighbor = np.random.choice(neighbors)
            fraction = np.random.rand()
            X_aug.append(fraction * X[instance, :] + (1.0 - fraction) * X[neighbor, :])
            y_aug.append(c)
    if X_aug:
        X = np.concatenate((X, X_aug))
        y = np.concatenate((y, y_aug))
    return X, y

X_aug, y_aug = smote(X, y)
X_aug.shape, y_aug.shape


In [ ]:

# Plot augmented data
plt.grid(True); plt.axhline(0.0, c='k'); plt.axvline(0.0, c='k')
plt.legend(*plt.scatter(X_aug[:, 0], X_aug[:, 1], c=y_aug, cmap=rgb_map, edgecolors='w').legend_elements(), title='Classes:')
plt.title('$y(x)$')
plt.axis('square'); plt.axis((-4.0, 4.0, -4.0, 4.0))
plt.show()


In [ ]:

# Using imbalanced-learn's SMOTE
from imblearn.over_sampling import SMOTE
model = SMOTE(k_neighbors=5)
X_aug, y_aug = model.fit_resample(X, y)
X_aug.shape, y_aug.shape


In [ ]:

# Plot SMOTE augmented data
plt.grid(True); plt.axhline(0.0, c='k'); plt.axvline(0.0, c='k')
plt.legend(*plt.scatter(X_aug[:, 0], X_aug[:, 1], c=y_aug, cmap=rgb_map, edgecolors='w').legend_elements(), title='Classes:')
plt.title('$y(x)$')
plt.axis('square'); plt.axis((-4.0, 4.0, -4.0, 4.0))
plt.show()


In [ ]:

# Learning curve
X_test, y_test = make_blobs(n_samples=[200, 300, 500], n_features=2, centers=[[-1, -1], [0, 1], [1, 0]])
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iterations, datasizes = 100, np.array([10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000])
metric_train, metric_test = np.empty((iterations, datasizes.size)), np.empty((iterations, datasizes.size))
for j, datasize in enumerate(datasizes):
    n_samples = (np.array([0.2, 0.3, 0.5]) * datasize).astype(int)
    for i in range(iterations):
        X_train, y_train = make_blobs(n_samples=n_samples, n_features=2, centers=[[-1, -1], [0, 1], [1, 0]])
        model = DecisionTreeClassifier(criterion='entropy', max_depth=3, min_samples_split=3)
        model.fit(X_train, y_train)
        metric_train[i, j] = accuracy_score(y_train, model.predict(X_train))
        metric_test[i, j] = accuracy_score(y_test, model.predict(X_test))

plt.plot(datasizes, metric_test.mean(axis=0), 'o-', label='test')
plt.plot(datasizes, metric_train.mean(axis=0), 'o-', label='train')
plt.fill_between(datasizes, metric_test.mean(axis=0) - metric_test.std(axis=0), metric_test.mean(axis=0) + metric_test.std(axis=0), color='C0', alpha=0.2)
plt.fill_between(datasizes, metric_train.mean(axis=0) - metric_train.std(axis=0), metric_train.mean(axis=0) + metric_train.std(axis=0), color='C1', alpha=0.2)
plt.xscale('log'); plt.ylim((0, 1))
plt.grid(True); plt.legend()
plt.title('Learning curve'); plt.xlabel('Training size'); plt.ylabel('Accuracy')
plt.show()


In [ ]:

# Validation curve for Decision Tree depth
iterations, max_depths = 100, np.array([1, 2, 5, 10, 20, 50, 100])
metric_train, metric_test = np.empty((iterations, max_depths.size)), np.empty((iterations, max_depths.size))
for j, max_depth in enumerate(max_depths):
    for i in range(iterations):
        X_train, y_train = make_blobs(n_samples=[200, 300, 500], n_features=2, centers=[[-1, -1], [0, 1], [1, 0]])
        model = DecisionTreeClassifier(criterion='entropy', max_depth=max_depth)
        model.fit(X_train, y_train)
        metric_train[i, j] = accuracy_score(y_train, model.predict(X_train))
        metric_test[i, j] = accuracy_score(y_test, model.predict(X_test))

plt.plot(max_depths, metric_test.mean(axis=0), 'o-', label='test')
plt.plot(max_depths, metric_train.mean(axis=0), 'o-', label='train')
plt.fill_between(max_depths, metric_test.mean(axis=0) - metric_test.std(axis=0), metric_test.mean(axis=0) + metric_test.std(axis=0), color='C0', alpha=0.2)
plt.fill_between(max_depths, metric_train.mean(axis=0) - metric_train.std(axis=0), metric_train.mean(axis=0) + metric_train.std(axis=0), color='C1', alpha=0.2)
plt.xscale('log'); plt.ylim((0, 1))
plt.grid(True); plt.legend()
plt.title('Validation curve'); plt.xlabel('Max depth'); plt.ylabel('Accuracy')
plt.show()
